# Datamine PLOTWS Process Example

This notebook demonstrates how to configure and run the **`plotws`** process wrapper in `dmstudio`.

## Process Description

## PLOTWS

Section plot in any direction through a wireframe model.

Scaling is completely automatic in this process, and any plot file or null prototype may be used; all that need be defined is the paper size.

PLOTWS produces an output plot file containing the intersection of a given plane with a wireframe model. The triangles of the wireframe model have not been intersected in any particular sequence, so there is no simple means in this program of generating an output perimeter.

Set section orientation as follows :-

## >>> SELECT ONE OF THE FOLLOWING --

## >>> 1 VERTICAL OR INCLINED SECTION >>> DEFINED ABOUT

## CENTRE POINT

## >>> 2 VERTICAL SECTION DEFINED BY >>> END POINTS

## >>> 3 PLAN VIEW DEFINED BY CORNER >>> POINTS

(>>> 4 SECTION FROM SECTION DEFINITION) (>>> FILE(if
defined) ) >>> Supply 1,2, or 3 (or 4) as appropriate.

If 1 selected, interaction as follows:

XC > X co-ordinate of centre of plane.
YC > Y co-ordinate of centre of plane.
ZC > Z co-ordinate of centre of plane.
DIP> Dip of section plane.
AZI> Azimuth of section plane.
HSIZE > Horizontal size of section plane.
VSIZE > Vertical size of section plane.

These are then echoed back, followed by the prompt

>ARE THESE CORRECT (Y/N) ? If any reply other than Y or y, interaction
repeated from the beginning.

If 2 selected, interaction as follows -

X1 > X co-ordinate of bottom left corner of section plane.

Y1 > Y co-ordinate of bottom left corner of section plane.

X2 > X co-ordinate of top right corner of section plane.

Y2 > Y co-ordinate of top right corner of section plane.

Z1 > Z co-ordinate of bottom left corner of section plane.

Z2 > Z co-ordinate of top right corner of section plane.

These are then echoed back, followed by the prompt

>ARE THESE CORRECT (Y/N) ? If any reply other than Y or y, interaction
repeated from the beginning.

If 3 selected, interaction as follows -

## >>> DEFINE XMIN,YMIN,XMAX, AND YMAX

X1 > X co-ordinate of bottom left corner of section plane.

Y1 > Y co-ordinate of bottom left corner of section plane.

X2 > X co-ordinate of top right corner of section plane.

Y2 > Y co-ordinate of top right corner of section plane.

ZC > Z co-ordinate of section plane.

These are then echoed back, followed by the prompt

>ARE THESE CORRECT (Y/N) ? If any reply other than Y or y, interaction
repeated from the beginning.

If 4 selected, interaction is as follows:

## >>> WHICH SECTION DO YOU WANT (OR ? FOR A LIST)

Enter SVALUE section number, or ? for a list of available sections.

### Input Files:

* **wiretr** (Wireframe Triangle):
  Input wireframe triangle file.
  Required=Yes

* **wirept** (Wireframe Points):
  Input wireframe point file.
  Required=Yes

* **proto** (Plot Prototype):
  Plot prototype file. Must contain the fields **X, Y, S1, S2** and **CODE** (numeric,
  explicit) and **XMIN, XMAX, YMIN, YMAX, XSCALE, YSCALE** (numeric, implicit). If these
  last 6 values set in **PROTO** , then corresponding parameters need not be set.
  Required=Yes

* **section** (Section Definition):
  Optional section definition file.
  Required=No

### Output Files:

* **plot** (Plot):
  Output plot file.
  Required=Yes

### Fields:

### Parameters:

* **linecode**:
  (1001) Line code for plotting - 1001 faint, or 1002 bold.
  Range=1001, 1006
  Values=Undefined
  Default=1001
  Required=No

* **frame**:
  (0) Set to 1 if frame required around plot.
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **charsize**:
  Character size in millimetres (4).
  Range=Undefined
  Values=Undefined
  Default=4
  Required=No

* **aspratio**:
  Aspect ratio, width / ht. for chars (0.9).
  Range=Undefined
  Values=Undefined
  Default=0.9
  Required=No

* **append**:
  Plot append flag. If set to 1 then the new plot will be appended to the **PLOT** file,
  if it exists and is a valid plot file (0).
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **xscale**:
  X scale in user data units per millimetre. If specified here or in **PROTO** this value
  will override section limits.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **yscale**:
  Y scale in user data units per millimetre. If specified here or in **PROTO** this value
  will override section limits.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **vertexag**:
  Controls vertical exaggeration. This must be set to allow different scales. The default
  is forced equal scales (1). = 0 allows different scales for both axes determined by
  **XSCALE** and **YSCALE** if provided or else by filling the data area to the section
  limits. > 0 sets value of **XSCALE** /**YSCALE**.
  Range=Undefined
  Values=Undefined
  Default=1
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('plotws')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.initialize_sandbox(notebook_folder, files_to_copy=files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute plotws
print("Running plotws...")
dm_cmd.plotws(
    wiretr_i='t_SurfaceTriangles',  # required
    wirept_i='t_SurfaceTriangles',  # required
    proto_i='t_mod1',  # required
    section_i='optional',  # required
    plot_o='t_plotws_out',  # required
    # linecode_p=1001,  # optional
    # frame_p=0,  # optional
    # charsize_p=4,  # optional
    # aspratio_p=0.9,  # optional
    # append_p=0,  # optional
    # xscale_p='optional',  # optional
    # yscale_p='optional',  # optional
    # vertexag_p=1,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("plotws execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_plotws_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")